# Data Cleaning

This notebook cleans the raw datasets from **The Movies Dataset** (Kaggle -- rounakbanik).

**Inputs** (`data/raw/`):
- `movies_metadata.csv` -- 45,466 movies with metadata
- `credits.csv` -- cast and crew JSON columns
- `keywords.csv` -- movie keyword/tag JSON columns
- `ratings.csv` -- 26M ratings from 270,896 users
- `links.csv` -- TMDB/IMDb ID cross-reference

**Outputs** (`data/processed/`):
- `movies_clean.csv`, `credits_clean.csv`, `keywords_clean.csv`,
  `ratings_clean.csv`, `links_clean.csv`

**Next:** `03_data_preprocessing.ipynb`

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

from src.data.data_loader import DataLoader
from src.data.data_cleaner import DataCleaner

print("Libraries loaded.")

Libraries loaded.


## Load Raw Data

In [2]:
loader = DataLoader()
cleaner = DataCleaner()

# Load raw datasets
print("Loading movies metadata...")
movies_raw = loader.load_movies_metadata()
print(f"  movies_metadata: {movies_raw.shape}")

print("Loading credits...")
credits_raw = loader.load_credits()
print(f"  credits: {credits_raw.shape}")

print("Loading keywords...")
keywords_raw = loader.load_keywords()
print(f"  keywords: {keywords_raw.shape}")

print("Loading ratings (this may take a moment)...")
ratings_raw = loader.load_ratings()
print(f"  ratings: {ratings_raw.shape}")

print("Loading links...")
links_raw = loader.load_links()
print(f"  links: {links_raw.shape}")

Loading movies metadata...
  movies_metadata: (45466, 24)
Loading credits...
  credits: (45476, 3)
Loading keywords...
  keywords: (46419, 2)
Loading ratings (this may take a moment)...
  ratings: (26024289, 4)
Loading links...
  links: (45843, 3)


## Clean Each Dataset

In [3]:
print("Cleaning movies metadata...")
movies_clean = cleaner.clean_movies_metadata(movies_raw)
print(f"  Before: {movies_raw.shape} -> After: {movies_clean.shape}")
print(f"  Columns: {list(movies_clean.columns)}")

Cleaning movies metadata...
  Before: (45466, 24) -> After: (42210, 28)
  Columns: ['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'year', 'month', 'genres_parsed', 'genre_names']


In [4]:
print("Cleaning credits...")
credits_clean = cleaner.clean_credits(credits_raw)
print(f"  Before: {credits_raw.shape} -> After: {credits_clean.shape}")

Cleaning credits...
  Before: (45476, 3) -> After: (45476, 7)


In [5]:
print("Cleaning keywords...")
keywords_clean = cleaner.clean_keywords(keywords_raw)
print(f"  Before: {keywords_raw.shape} -> After: {keywords_clean.shape}")

Cleaning keywords...
  Before: (46419, 2) -> After: (46419, 4)


In [6]:
print("Cleaning ratings (26M rows — takes ~60s)...")
ratings_clean = cleaner.clean_ratings(ratings_raw)
print(f"  Before: {ratings_raw.shape} -> After: {ratings_clean.shape}")
print(f"  Rating range: {ratings_clean.rating.min():.1f} – {ratings_clean.rating.max():.1f}")
print(f"  Unique users: {ratings_clean.userId.nunique():,}")
print(f"  Unique movies: {ratings_clean.movieId.nunique():,}")

Cleaning ratings (26M rows — takes ~60s)...
  Before: (26024289, 4) -> After: (26024289, 4)
  Rating range: 0.5 – 5.0
  Unique users: 270,896
  Unique movies: 45,115


In [7]:
print("Cleaning links...")
links_clean = cleaner.clean_links(links_raw)
print(f"  Before: {links_raw.shape} -> After: {links_clean.shape}")

Cleaning links...
  Before: (45843, 3) -> After: (45624, 3)


## Save Cleaned Data

In [8]:
cleaner.save_cleaned_data(
    movies_clean=movies_clean,
    credits_clean=credits_clean,
    keywords_clean=keywords_clean,
    ratings_clean=ratings_clean,
    links_clean=links_clean,
)
print("All cleaned datasets saved to data/processed/")
print("Next: run 03_data_preprocessing.ipynb")

Cleaned data saved successfully!
All cleaned datasets saved to data/processed/
Next: run 03_data_preprocessing.ipynb


## Summary

| Dataset | Raw rows | Clean rows | Key changes |
|---|---|---|---|
| movies_metadata | 45,466 | 42,210 | Removed non-numeric IDs; parsed genres JSON; added year/month columns |
| credits | 45,476 | 45,476 | Parsed cast/crew JSON; extracted director, top-3 cast names |
| keywords | 46,419 | 46,419 | Parsed keyword JSON; extracted keyword name list |
| ratings | 26,024,289 | 26,024,289 | Validated range 0.5--5.0; 270,896 unique users; 45,115 unique movies |
| links | 45,843 | 45,624 | Removed movies without valid TMDB IDs |

All cleaned files are saved to `data/processed/` and consumed by `03_data_preprocessing.ipynb`.